<a href="https://colab.research.google.com/github/Aestivation/CNN-LSTM-Model/blob/branch1/CNN%2BLSTM_Model_Applying_on_Lyft_Motion_Prediction_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kneroma/lyft-motion-prediction-autonomous-vehicles-as-csv")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

# مسیر فایل‌ها در Colab (پس از دانلود از Kaggle)
dataset_path = "/root/.cache/kagglehub/datasets/kneroma/lyft-motion-prediction-autonomous-vehicles-as-csv/versions/1"

# لیست فایل‌های داخل مجموعه داده
files = os.listdir(dataset_path)
print("📂 Files in Dataset:", files)

# بررسی یک فایل CSV
sample_file = os.path.join(dataset_path, files[0])  # اولین فایل CSV
df = pd.read_csv(sample_file)
print(df.head())  # نمایش اولین ردیف‌های


In [ ]:
# بررسی فایل‌های مربوط به خودروها
agents_file = os.path.join(dataset_path, 'agents_0_10019001_10019001.csv')  # تغییر نام بر اساس فایل شما
df_agents = pd.read_csv(agents_file)

print(df_agents.head())  # نمایش چند سطر اول


In [ ]:
import pandas as pd
import numpy as np

# بارگذاری داده‌های خودروها
agents_file = os.path.join(dataset_path, 'agents_0_10019001_10019001.csv')
df_agents = pd.read_csv(agents_file)

# انتخاب ویژگی‌های کلیدی برای مدل
features = ['centroid_x', 'centroid_y', 'velocity_x', 'velocity_y', 'yaw']
sequence_length = 10  # 10 فریم متوالی برای مدل LSTM

# مرتب‌سازی داده‌ها بر اساس شناسه خودرو و فریم
df_agents = df_agents.sort_values(by=['track_id', 'frame_db_id'])

# ایجاد دنباله‌های زمانی برای LSTM
def create_sequences(data, sequence_length):
    sequences, targets = [], []
    for track_id in data['track_id'].unique():
        track_data = data[data['track_id'] == track_id]  # داده‌های مربوط به هر وسیله نقلیه
        track_data = track_data[features].values

        for i in range(len(track_data) - sequence_length):
            seq = track_data[i:i+sequence_length]  # ۱۰ فریم پشت سر هم
            target = track_data[i+sequence_length][:2]  # پیش‌بینی موقعیت x, y
            sequences.append(seq)
            targets.append(target)

    return np.array(sequences), np.array(targets)

X_train, Y_train = create_sequences(df_agents, sequence_length)

print("✅ شکل داده‌های ورودی:", X_train.shape)  # (نمونه‌ها, ۱۰, ویژگی‌ها)
print("✅ شکل داده‌های خروجی:", Y_train.shape)  # (نمونه‌ها, ۲)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.utils import plot_model



model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(sequence_length, len(features))),
    BatchNormalization(),
    LSTM(64, return_sequences=True),  # نیازی به Reshape نیست
    Dropout(0.2),
    LSTM(32),
    Dense(64, activation="relu"),
    Dense(2, activation="linear")  # خروجی X, Y آینده
])

model.compile(optimizer="RMSprop", loss="mse", metrics=["mae"])
model.summary()

plot_model(model, to_file="cnn_model.png", show_shapes=True, show_layer_names=True)

from IPython.display import Image

Image("cnn_model.png")




In [ ]:
# تقسیم داده‌ها به آموزش و اعتبارسنجی
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)

import numpy as np



# فقط ۲۰٪ داده‌ها را نگه داریم

sample_fraction = 0.2

sample_indices = np.random.choice(len(X_train), int(len(X_train) * sample_fraction), replace=False)



X_train_small = X_train[sample_indices]

Y_train_small = Y_train[sample_indices]



# اعمال همین کار برای داده‌های تست

sample_indices_test = np.random.choice(len(X_val), int(len(X_val) * sample_fraction), replace=False)

X_val_small = X_val[sample_indices_test]

Y_val_small = Y_val[sample_indices_test]



print(f"✅ New Training Data Shape: {X_train_small.shape}")

print(f"✅ New Testing Data Shape: {X_val_small.shape}")

from tensorflow.keras.callbacks import EarlyStopping



# تعریف Early Stopping

early_stopping = EarlyStopping(

    monitor='val_loss',    # متریک مورد بررسی (می‌تواند 'val_mae' هم باشد)

    patience=5,            # تعداد ایپوک‌هایی که صبر می‌کند قبل از توقف

    restore_best_weights=True # وزن‌های بهترین مدل را ذخیره کند

)


# آموزش مدل
history = model.fit(
    X_train_small, Y_train_small,
    validation_data=(X_val_small, Y_val_small),
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping]
)


In [ ]:
import matplotlib.pyplot as plt

# رسم نمودار خطای MSE
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.title("Training vs Validation Loss")
plt.show()

# رسم نمودار MAE
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error')
plt.legend()
plt.title("Training vs Validation MAE")
plt.show()


In [ ]:
# پیش‌بینی روی داده‌های تستی
sample_input = X_val[:5]  # گرفتن چند نمونه از داده‌های اعتبارسنجی
predictions = model.predict(sample_input)

# نمایش نتایج
print("پیش‌بینی مدل برای مختصات آینده:")
for i in range(len(predictions)):
    print(f"نمونه {i+1}: X={predictions[i][0]:.2f}, Y={predictions[i][1]:.2f}")


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# پیش‌بینی روی داده‌های تستی
Y_pred = model.predict(X_val)

# محاسبه خطاها
mse = mean_squared_error(Y_val, Y_pred)
mae = mean_absolute_error(Y_val, Y_pred)

print(f"MSE (Mean Squared Error): {mse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")

print("میانگین مقدار Y_train:", np.mean(Y_train))

print("بازه مقادیر Y_train:", np.min(Y_train), "تا", np.max(Y_train))
